In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU не выделена")
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "T4" not in name and "A100" not in name and "H100" not in name:
    raise SystemExit(f"Нужна T4: {name} несовместима с предустановленным PyTorch")

modules = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
for path in glob.glob(os.path.dirname(modules[0]) + "/*.py"):
    shutil.copy(path, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

# Модель-разметчик берётся из вывода предыдущего ядра, а не заливается датасетом:
# 678 MB весов уже лежат на Kaggle, дублировать их незачем.
labeler = os.path.dirname(glob.glob("/kaggle/input/**/inference_config.json", recursive=True)[0])
print("разметчик:", labeler)
amb_pairs = glob.glob("/kaggle/input/**/amb_pairs.parquet", recursive=True)[0]
amb_texts = glob.glob("/kaggle/input/**/amb_texts.parquet", recursive=True)[0]
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])

sys.argv = ["pseudo_label", "--model", labeler, "--pairs", amb_pairs,
            "--texts", amb_texts, "--texts", pack + "/item_texts.parquet",
            "--out", "/kaggle/working/pseudo_pairs.parquet", "--batch-size", "256"]
from src.pseudo_label import main as label_main
label_main()

os.makedirs("/kaggle/working/pack", exist_ok=True)
for path in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(path)
    if not os.path.exists(dst): os.symlink(path, dst)

sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            "--base-model", "DeepPavlov/rubert-base-cased",
            "--extra-pairs", "/kaggle/working/pseudo_pairs.parquet",
            "--extra-texts", amb_texts,
            "--epochs", "1", "--batch-size", "256", "--max-length", "256",
            "--output", "/kaggle/working/ce_self"]
from src.train_ce_large import main as train_main
train_main()
